# 09. 사진 보고 말하기 — VLM (openvino-genai)

VLM(Vision Language Model)은 사진과 글을 함께 받아 글로 답한다. 여기서는 **Gemma3 4B (INT4)** 를 쓴다.
토큰화 · 생성 반복 · 이미지 처리는 `openvino_genai.VLMPipeline` 이 한다 — 이 단원은 **라이브러리를 잘 쓰는 법**을 본다.

- 모델이 3.5GB 라 처음 올리는 데 시간이 걸린다. edge-lab(`run.bat`)이 켜져 있으면 끄고 한다 (메모리 두 벌).
- `CACHE_DIR` 을 주면 두 번째부터 훨씬 빨리 올라간다.

## 1. 모델 올리기

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import openvino_genai as og

VLM_DIR = MODELS / "vlm" / "gemma3-4b-int4"
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
CACHE = Path("ov-cache"); CACHE.mkdir(exist_ok=True)

t0 = time.perf_counter()
pipe = og.VLMPipeline(str(VLM_DIR), DEVICE, CACHE_DIR=str(CACHE))
print(f"올림: {time.perf_counter() - t0:.0f}초 ({DEVICE})")

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 묻고 답 받기 — 한 글자씩 흘려 받기(streaming)
사진은 `ov.Tensor` 로 넣는다: `[1, 세로, 가로, 3]` RGB uint8. 너무 크면 느리기만 하다 — 긴 쪽 640 이 알맞다.

In [ ]:
def to_tensor(bgr, max_side=640):
    h, w = bgr.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s < 1.0:
        bgr = cv2.resize(bgr, (int(w * s), int(h * s)))
    return ov.Tensor(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)[None].astype(np.uint8))

cfg = og.GenerationConfig()
cfg.max_new_tokens = 160
cfg.do_sample = False                 # 같은 질문엔 같은 답
cfg.repetition_penalty = 1.15         # 작은 모델의 같은 말 반복 막기

RUNNING = og.StreamingStatus.RUNNING if hasattr(og, "StreamingStatus") else False
first = {}
def streamer(piece):
    first.setdefault("t", time.perf_counter())
    print(piece, end="", flush=True)
    return RUNNING                    # 계속 만들어라

def ask(bgr, question):
    first.clear()
    t0 = time.perf_counter()
    out = pipe.generate(question, image=to_tensor(bgr), generation_config=cfg, streamer=streamer)
    t1 = time.perf_counter()
    n = len(pipe.get_tokenizer().encode(str(out)).input_ids.data[0])
    print(f"\n\n첫 글자까지 {first.get('t', t1) - t0:.1f}초 · 전체 {t1 - t0:.1f}초 · 약 {n / max(t1 - first.get('t', t0), 1e-6):.1f} 토큰/초")
    return str(out)

answer = ask(frame, "이 사진을 한국어로 두 문장으로 설명하세요.")

In [ ]:
answer = ask(frame, "사진 속 물건을 쉼표로 나열하세요. 설명은 쓰지 마세요.")

## 4. 사진 없이 글만 — 대화
같은 모델에 사진을 안 주면 글 LLM 처럼 쓸 수 있다 (08 번 RAG 가 이렇게 쓴다).

In [ ]:
out = pipe.generate("OpenVINO 의 NPU 가 무엇인지 한 문장으로 설명하세요.", generation_config=cfg, streamer=streamer)

## 해 볼 것
1. `cfg.do_sample = True; cfg.temperature = 0.8` 로 같은 질문을 두 번 해 보자.
2. `max_side` 를 320 · 896 으로 바꿔 속도와 답을 비교해 보자.
3. "사진 속 사람 수를 숫자만 답하세요" 처럼 **형식을 정해** 물으면 코드에서 답을 쓰기 쉬워진다. JSON 으로 답하게 해 보자.
4. `DEVICE` 를 `"CPU"` 로 바꾸면 토큰/초가 얼마나 줄어드나?